This notebook incrementally calculates profit estimates of performing skill based actions on an hourly basis and writes the data to  
Table xxx

In [1]:
import pyspark.sql.functions as sf
from time import time

Enriched Data

In [2]:
# Read data from 'runescape.02_silver.1h_prices_enriched'
df_1h_prices = spark.read.table("runescape.02_silver.1h_prices_enriched")
df_1h_prices = df_1h_prices.drop("highalch","limit","members")

In [3]:
# get skill input data
df_skill_inputs = spark.read.table("runescape.02_silver.skill_action_inputs_enriched")

In [4]:
# Read data from 'runescape.02_silver.1h_prices_last_enriched'
df_1h_prices_last = spark.read.table("runescape.02_silver.1h_prices_last_enriched")
# drop time column
df_1h_prices_last = df_1h_prices_last.drop("time", "highalch", "limit", "members")

In [5]:
# TODO replace the code below with code that gets the most recent data from 'runescape.02_silver.1h_prices_enriched'
# instead of using the time window
# will need to make sure that the merge works when writing to gold layer.

In [6]:
# filter data to only last 90 minutes (extra time allowed for any delays)
# job will run this notebook every hour
unix_time_window = time() - 11400 # 90 minutes
df_1h_prices_latest = df_1h_prices.filter(f"time > {unix_time_window}")
# validate there is only 1 value for time in df_1h_prices_latest
time_range = df_1h_prices_latest.select("time").distinct().count()
if time_range > 1:
    raise Exception("There is more than 1 time present in df_1h_prices_latest")
# df with the time for this data set.  This will be used later for the imputed data.
df_time = df_1h_prices_latest.select(sf.first_value("time").alias("time"))
# check to make sure the time value is not null
if df_time.collect()[0][0] is None:
    raise Exception("The time in df_time is Null")

In [7]:
df_1h_prices_latest.show()

+---+------------+------------------+-----------+-----------------+----------+------------------+
| id|avg1HourHigh|avg1HourHighVolume|avg1HourLow|avg1HourLowVolume|      time|              name|
+---+------------+------------------+-----------+-----------------+----------+------------------+
|  2|         297|            758547|        292|           336399|1772636400|  Steel cannonball|
| 30|         800|                36|        586|               14|1772636400|     Bucket of wax|
| 36|         369|                21|        367|               13|1772636400|            Candle|
| 39|           5|             15215|          4|             3280|1772636400|  Bronze arrowtips|
| 40|           7|             20550|          7|             2085|1772636400|    Iron arrowtips|
| 41|          18|             21024|         18|             4296|1772636400|   Steel arrowtips|
| 42|          39|              1804|         38|             1700|1772636400| Mithril arrowtips|
| 43|          90|  

Need to create a dataframe for all skill action outputs (and prices/ volumes) and thier respective inputs (and prices/ volumes)  
This dataframe also should have price data (and inputs) for items that did not have data in the last hour and instead use historic data from table 1h_prices_last_enriched

In [8]:
# get df with all distinct skill action output and input id's
# cross join with df_time 
# TODO consider how this is impacted by implementation of other skills
# TODO consider renaming df_outputs_time to something less dumb
df_outputs_time = df_skill_inputs\
    .select("id_output")\
    .union(df_skill_inputs\
    .select("id_input"))\
    .dropDuplicates()\
    .withColumnRenamed("id_output","id")\
    .crossJoin(df_time)


Data imputation for missing/ filtered out data.  
Imputation method: Use the last known avg values based on hourly data in table 'runescape.02_silver.1h_prices_last_enriched'  
This approach has it's limiations and should be reconsidered.

In [9]:
# create df with imputed price data using the prices from 1h_prices_last_enriched
# This data will be used when there is no data for the respective item for profit calcs
df_prices_imputed = df_1h_prices_last.join(df_outputs_time, "id")

In [10]:
# create df with only items that have missing 'real' price data
df_prices_missing = df_prices_imputed.join(df_1h_prices_latest.select("id"),"id","left_anti")

In [11]:
df_prices_missing.sort("id").show()

+----+------------+------------------+-----------+-----------------+--------------------+----------+
|  id|avg1HourHigh|avg1HourHighVolume|avg1HourLow|avg1HourLowVolume|                name|      time|
+----+------------+------------------+-----------+-----------------+--------------------+----------+
|1063|           3|                70|          3|               61|   Leather vambraces|1772636400|
|1131|          21|                13|         13|              138|    Hardleather body|1772636400|
|1133|         352|                12|        304|               26|        Studded body|1772636400|
|1399|        8823|              2460|       8773|             2027|   Earth battlestaff|1772636400|
|1673|         160|             36000|        159|             1630|     Gold amulet (u)|1772636400|
|1675|         436|              2141|        419|               34| Sapphire amulet (u)|1772636400|
|1677|         639|                15|        630|              343|  Emerald amulet (u)|17

In [12]:
# union 1h_price data to get complete data set
df_1h_prices_full = df_prices_missing.unionByName(df_1h_prices_latest)

In [13]:
df_1h_prices_latest.sort("id").show()

+---+------------+------------------+-----------+-----------------+----------+------------------+
| id|avg1HourHigh|avg1HourHighVolume|avg1HourLow|avg1HourLowVolume|      time|              name|
+---+------------+------------------+-----------+-----------------+----------+------------------+
|  2|         297|            758547|        292|           336399|1772636400|  Steel cannonball|
| 30|         800|                36|        586|               14|1772636400|     Bucket of wax|
| 36|         369|                21|        367|               13|1772636400|            Candle|
| 39|           5|             15215|          4|             3280|1772636400|  Bronze arrowtips|
| 40|           7|             20550|          7|             2085|1772636400|    Iron arrowtips|
| 41|          18|             21024|         18|             4296|1772636400|   Steel arrowtips|
| 42|          39|              1804|         38|             1700|1772636400| Mithril arrowtips|
| 43|          90|  

In [14]:
df_1h_prices_full.sort("id").show()

+---+------------+------------------+-----------+-----------------+------------------+----------+
| id|avg1HourHigh|avg1HourHighVolume|avg1HourLow|avg1HourLowVolume|              name|      time|
+---+------------+------------------+-----------+-----------------+------------------+----------+
|  2|         297|            758547|        292|           336399|  Steel cannonball|1772636400|
| 30|         800|                36|        586|               14|     Bucket of wax|1772636400|
| 36|         369|                21|        367|               13|            Candle|1772636400|
| 39|           5|             15215|          4|             3280|  Bronze arrowtips|1772636400|
| 40|           7|             20550|          7|             2085|    Iron arrowtips|1772636400|
| 41|          18|             21024|         18|             4296|   Steel arrowtips|1772636400|
| 42|          39|              1804|         38|             1700| Mithril arrowtips|1772636400|
| 43|          90|  

Build table with profit calculations

In [15]:
# create df with full price data set combined with inputs (qty, id's)
# this will be used to calculate the input cost for each skill action
df_1h_prices_full_w_inputs = df_1h_prices_full.join(df_skill_inputs,
                                            df_1h_prices_full.id == df_skill_inputs.id_output,
                                            "left_outer")
df_1h_prices_full_w_inputs = df_1h_prices_full_w_inputs.drop('id','name','highalch','limit','members')
# rename columns to be associated with output data
df_1h_prices_full_w_inputs = df_1h_prices_full_w_inputs.withColumnsRenamed(
    {'avg1HourHigh':'avg_high_output',
     'avg1HourHighVolume':'avg_high_volume_output',
     'avg1HourLow':'avg_low_output',
     'avg1HourLowVolume':'avg_low_volume_output'}
)

In [16]:
df_1h_prices_full_w_inputs.show()

+---------------+----------------------+--------------+---------------------+----------+-------------------+---------+------------------+--------+---------+--------+
|avg_high_output|avg_high_volume_output|avg_low_output|avg_low_volume_output|      time|             output|input_qty|             input|   skill|id_output|id_input|
+---------------+----------------------+--------------+---------------------+----------+-------------------+---------+------------------+--------+---------+--------+
|              2|                   116|             2|                  332|1772636400|     Empty oil lamp|      1.0|      Molten glass|crafting|     4525|    1775|
|            660|                  8382|           651|                 1045|1772636400|      Opal bracelet|      1.0|        Silver bar|crafting|    21117|    2355|
|           3316|                  1002|          3333|                  125|1772636400|     Topaz bracelet|      1.0|        Silver bar|crafting|    21123|    2355|
|   

In [17]:
# join to get input price data
df_1h_prices_full_combined = df_1h_prices_full_w_inputs.join(df_1h_prices_full,
                                                                     [df_1h_prices_full_w_inputs.id_input == df_1h_prices_full.id,
                                                                      df_1h_prices_full_w_inputs.time == df_1h_prices_full.time],
                                                                     "left_outer").drop(df_1h_prices_full_w_inputs.time)

In [18]:
df_1h_prices_full_combined.show()

+---------------+----------------------+--------------+---------------------+-------------------+---------+------------------+--------+---------+--------+-----+------------+------------------+-----------+-----------------+------------------+----------+
|avg_high_output|avg_high_volume_output|avg_low_output|avg_low_volume_output|             output|input_qty|             input|   skill|id_output|id_input|   id|avg1HourHigh|avg1HourHighVolume|avg1HourLow|avg1HourLowVolume|              name|      time|
+---------------+----------------------+--------------+---------------------+-------------------+---------+------------------+--------+---------+--------+-----+------------+------------------+-----------+-----------------+------------------+----------+
|              2|                   116|             2|                  332|     Empty oil lamp|      1.0|      Molten glass|crafting|     4525|    1775| 1775|          97|             36678|         96|            40492|      Molten glass|

In [19]:
# calculate input costs
df_input_cost = df_1h_prices_full_combined.withColumns({'input_cost_low': sf.round(sf.col("avg1HourLow")*sf.col("input_qty"),1),
                                                            'input_cost_high': sf.round(sf.col("avg1HourHigh")*sf.col("input_qty"),1)})

In [20]:
df_input_total_cost = df_input_cost.groupBy("time","output", "skill").sum("input_cost_high", "input_cost_low")
df_input_total_cost = df_input_total_cost.withColumnsRenamed({"sum(input_cost_high)": "total_input_cost_high"
                                                              , "sum(input_cost_low)": "total_input_cost_low"})


In [21]:
tax = .98 #simplified tax for now, can be implemented as function
df_1h_profits = (df_1h_prices_full_combined.join(df_input_total_cost, ["time", "output", "skill"])\
    .drop("name")\
    # TODO rename input cols
    # calculate profit estimates without tax
    # TODO add tax and implement as function
    .withColumns({
        "profit_low_est": sf.round(tax*sf.col("avg_low_output") - sf.col("total_input_cost_high"),2),
        "profit_high_est": sf.round(tax*sf.col("avg_high_output") - sf.col("total_input_cost_low"),2)
    }))

In [ ]:
df_1h_profits.show()